# 5.1 — Correlation Analysis: Latent Dimensions × Clinical Factors
**Thesis:** Unsupervised Deep Learning for Parkinson's Disease Imaging  
**Author:** Natnael Solomon Gebremichael — UniMiB MSc Data Science  

## What this notebook does
1. Loads the clean train/val files from notebook 5.0  
2. Filters to active latent dimensions (KL/variance threshold)  
3. Tests correlation between each active dimension and 12 clinical/demographic factors  
4. Applies Bonferroni correction for multiple comparisons  
5. Validates significant findings on the patient-stratified held-out set  
6. Runs ANCOVA to separate scanner effects from biological signal  
7. Produces the correlation heatmap and ANCOVA scatter plot  

## Improvements over baseline (Mahmoud's 5.1.1)
- 12 factors tested vs 8 — adds UPDRS I–IV, MoCA, RBD score, Hoehn & Yahr  
- SBR PCA loaded from saved objects — not refitted  
- Patient-stratified validation set — no data leakage  
- Full biological covariates in ANCOVA — not just SBR_PC1  


## 1. Import and Configuration

In [7]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
warnings.filterwarnings('ignore')


# File paths
TRAIN_FILE = '../../data/processed/clinical_merged/train.csv'
VAL_FILE = '../../data/processed/clinical_merged/val.csv'
SCALER_PATH = '../../results/models/scaler_sbr.pkl'
PCA_PATH = '../../results/models/pca_sbr.pkl'

OUTPUT_DIR = '../../results/correlation'
FIGURES_DIR = '../../results/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Parameters
PATIENT_COL        = 'PATNO'
LABEL_COL          = 'label'
LATENT_COLS        = [f'latent_{i}' for i in range(256)]
VARIANCE_THRESHOLD = 0.1    # dims with std below this = collapsed
ALPHA              = 0.05   # base significance level
VAL_ALPHA          = 0.1    # lenient threshold for validation confirmation


# Clinical Factors
 # Continuous -> Pearson r^2
CONTINUOUS_FACTORS = [
    'AGE_AT_VISIT',
    'SBR_PC1', 'SBR_PC2', 'SBR_PC3',
    'UPDRS1_TOTAL', 'UPDRS2_TOTAL', 'UPDRS3_TOTAL', 'UPDRS4_TOTAL',
    'MOCA_TOTAL', 'RBD_SCORE',
]

 # Categorical
CATEGORICAL_FACTORS = ['SEX', 'HANDED', 'Manufacturer', 'HOEHN_YAHR']

ALL_FACTORS = CONTINUOUS_FACTORS + CATEGORICAL_FACTORS

print("Configuration loaded.")
print(f"  Continuous factors:  {len(CONTINUOUS_FACTORS)}")
print(f"  Categorical factors: {len(CATEGORICAL_FACTORS)}")
print(f"  Total factors:       {len(ALL_FACTORS)}")

Configuration loaded.
  Continuous factors:  10
  Categorical factors: 4
  Total factors:       14


## 2. Load Data

In [9]:
df_train = pd.read_csv(TRAIN_FILE)
df_val   = pd.read_csv(VAL_FILE)

# Load pre-fitted SBR PCA objects
with open(SCALER_PATH, 'rb') as f: scaler_sbr = pickle.load(f)
with open(PCA_PATH,    'rb') as f: pca_sbr    = pickle.load(f)

print("Data loaded")
print(f"Train: {df_train.shape}, {df_train[PATIENT_COL].nunique()} patients")
print(f"Val:   {df_val.shape},   {df_val[PATIENT_COL].nunique()} patients")

print(f"\nLabel distribution (train):")
print(df_train[LABEL_COL].value_counts().to_string())

print(f"\nFactor availability in train:")
for f in ALL_FACTORS:
    if f in df_train.columns:
        pct = df_train[f].notna().mean() * 100
        print(f"  {f:<25} {pct:.1f}%")
    else:
        print(f"  {f:<25} MISSING")

Data loaded
Train: (2221, 315), 1148 patients
Val:   (545, 315),   289 patients

Label distribution (train):
label
PD         1948
Control     185
SWEDD        88

Factor availability in train:
  AGE_AT_VISIT              99.9%
  SBR_PC1                   100.0%
  SBR_PC2                   100.0%
  SBR_PC3                   100.0%
  UPDRS1_TOTAL              67.4%
  UPDRS2_TOTAL              67.4%
  UPDRS3_TOTAL              67.3%
  UPDRS4_TOTAL              39.9%
  MOCA_TOTAL                73.8%
  RBD_SCORE                 48.5%
  SEX                       99.9%
  HANDED                    99.9%
  Manufacturer              100.0%
  HOEHN_YAHR                67.3%


## 3. Filter to Activate Latent Dimension